# Explore Classification Results

Interactive notebook for inspecting and comparing classifier training runs.
Mirrors `05.plot_classifier_results.py` but designed for cell-by-cell exploration with immediate plot feedback.

**Sections**
1. Setup & configuration
2. Single-run analysis (summary, training curves, metric distributions, confusion matrix)
3. Comparative ROC curves across multiple runs

## 1. Setup

In [ ]:
%matplotlib inline
import os, pickle, itertools, math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.preprocessing import label_binarize
from matplotlib.colors import LinearSegmentedColormap

import sys
sys.path.insert(0, '/users/mbredber/p2_DCRECLASS/src')

from dcreclass.data import get_classes
from dcreclass.utils.calc_tools import (
    recalculate_metrics_with_correct_positive_class, round_to_1
)
from dcreclass.utils.plotting import (
    plot_training_history, plot_avg_std_confusion_matrix
)

plt.rcParams.update({'font.size': 9})

OUTPUT_DIR = '/users/mbredber/p2_DCRECLASS/outputs/scratch/figures/classifying/explore_classification_results_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Imports OK')
print('Output dir:', OUTPUT_DIR)

In [ ]:
@dataclass
class RunConfig:
    """All parameters that identify one training run.
    Defaults match script 04.
    """
    classifier:      str   = 'ImageCNN'
    version:         str   = 'RAW'          # e.g. RAW, T25kpc, T50kpc, T100kpc
    crop_mode:       str   = 'beam_crop'    # beam_crop | beam_crop_no_sub | fov_crop | cheat_crop
    blur_method:     str   = 'circular'     # circular | circular_no_sub | cheat
    lr:              float = 4e-5
    reg:             float = 1e-1
    label_smoothing: float = 0.1
    percentile_lo:   int   = 30
    percentile_hi:   int   = 99
    folds:           List[int] = field(default_factory=lambda: [0])
    num_experiments: int   = 1
    galaxy_classes:  List[int] = field(default_factory=lambda: [50, 51])
    crop_size:       Tuple[int, int] = (512, 512)
    downsample_size: Tuple[int, int] = (128, 128)
    global_norm_mode: str  = 'percentile'
    adjust_positive_class: bool = True
    data_run_dir:    str   = '/users/mbredber/p2_DCRECLASS/outputs/scratch'

    @property
    def metrics_dir(self) -> str:
        sub = (f"{self.classifier}_{self.crop_mode}_{self.blur_method}_"
               f"{self.lr}_{self.reg}_{self.percentile_lo}_"
               f"{self.percentile_hi}_{self.label_smoothing}")
        return os.path.join(self.data_run_dir, 'data', 'metrics', sub)

    @property
    def dataset_sizes(self) -> Dict[int, List[int]]:
        if self.galaxy_classes == [50, 51]:
            return {fold: [3000] for fold in range(10)}
        if self.galaxy_classes == [52, 53]:
            return {fold: [2, 16, 168] for fold in range(10)}
        raise ValueError(f'No dataset_sizes defined for galaxy_classes={self.galaxy_classes}')

    def base_key(self, fold: int, subset_size: int) -> str:
        """Canonical key — mirrors script 04's _base()."""
        return (f"{self.classifier}_ver{self.version}_cm{self.crop_mode}"
                f"_lr{self.lr}_reg{self.reg}_ls{self.label_smoothing}"
                f"_lo{self.percentile_lo}_hi{self.percentile_hi}"
                f"_f{fold}_ss{round_to_1(subset_size)}")

    def pkl_path(self, fold: int, subset_size: int, experiment: int) -> str:
        return os.path.join(
            self.metrics_dir,
            f"{self.base_key(fold, subset_size)}_e{experiment}.pkl"
        )

print('RunConfig defined')

In [ ]:
def load_run(cfg: RunConfig, verbose: bool = True) -> dict:
    """Load all pkl files for a RunConfig.

    Returns a flat metrics dict (keyed as '{metric}_{subset}_{fold}_{exp}_{lr}_{reg}')
    plus entries for raw label/prob dicts and history, mirroring script 05.
    Special keys: '_cluster' (cluster metrics dict), '_cfg' (the RunConfig).
    """
    tot = {}
    cluster = {'errors': [], 'distances': [], 'std_devs': []}
    loaded, failed = 0, 0
    _cluster_map = {
        'cluster_error': 'errors',
        'cluster_distance': 'distances',
        'cluster_std_dev': 'std_devs',
    }

    for experiment, fold in itertools.product(range(cfg.num_experiments), cfg.folds):
        for subset_size in cfg.dataset_sizes[fold]:
            path = cfg.pkl_path(fold, subset_size, experiment)
            try:
                with open(path, 'rb') as fh:
                    data = pickle.load(fh)
            except FileNotFoundError:
                if verbose:
                    print(f'  [miss] {os.path.basename(path)}')
                failed += 1
                continue
            except Exception as exc:
                if verbose:
                    print(f'  [err ] {os.path.basename(path)}: {exc}')
                failed += 1
                continue

            for src_key, dst_list in _cluster_map.items():
                val = data.get(src_key)
                if val is not None:
                    cluster[dst_list].append(val)

            base = cfg.base_key(fold, subset_size)
            y_true  = data['all_true_labels'].get(base, [])
            y_pred  = data['all_pred_labels'].get(base, [])
            y_probs = data['all_pred_probs'].get(base, [])

            if not y_true or not y_pred:
                failed += 1
                continue

            if cfg.adjust_positive_class:
                acc, prec, rec, f1 = recalculate_metrics_with_correct_positive_class(
                    y_true, y_pred, pos_label=0
                )
            else:
                m = data['metrics']
                acc  = m.get('accuracy',  [0.0])[0]
                prec = m.get('precision', [0.0])[0]
                rec  = m.get('recall',    [0.0])[0]
                f1   = m.get('f1_score',  [0.0])[0]

            auc_val = float('nan')
            if y_probs:
                try:
                    p = np.asarray(y_probs)
                    scores = p[:, 1] if p.ndim == 2 and p.shape[1] > 1 else p.ravel()
                    if np.unique(np.asarray(y_true)).size >= 2:
                        fpr_, tpr_, _ = roc_curve(y_true, scores)
                        auc_val = auc(fpr_, tpr_)
                except Exception:
                    pass

            k = f'{subset_size}_{fold}_{experiment}_{cfg.lr}_{cfg.reg}'
            for metric, val in [('accuracy', acc), ('precision', prec),
                                 ('recall', rec), ('f1_score', f1), ('auc', auc_val)]:
                tot.setdefault(f'{metric}_{k}', []).append(val)

            for store_key, obj in [
                ('all_true_labels', data['all_true_labels']),
                ('all_pred_labels', data['all_pred_labels']),
                ('all_pred_probs',  data['all_pred_probs']),
                ('history',         data['history']),
                ('training_times',  data['training_times']),
            ]:
                tot.setdefault(f'{store_key}_{k}', []).append(obj)

            loaded += 1

    print(f'Loaded {loaded} pkl files, failed/missing {failed}.')
    tot['_cluster'] = cluster
    tot['_cfg'] = cfg
    return tot

print('load_run() defined')

## 2. Single-run analysis

In [ ]:
# ── Edit this cell to point at the run you want to inspect ───────────────────
cfg = RunConfig(
    classifier      = 'ImageCNN',
    version         = 'RAW',
    crop_mode       = 'beam_crop',
    blur_method     = 'circular',
    lr              = 4e-5,
    reg             = 1e-1,
    folds           = list(range(10)),
    num_experiments = 1,
)

print('metrics_dir :', cfg.metrics_dir)
print('example pkl :', cfg.pkl_path(0, 3000, 0))
print('exists      :', os.path.exists(cfg.pkl_path(0, 3000, 0)))

In [ ]:
metrics = load_run(cfg)

# Aggregate scalar metrics across all loaded folds / experiments
metrics_last = defaultdict(list)
for fold in cfg.folds:
    subset = max(cfg.dataset_sizes[fold])
    for exp in range(cfg.num_experiments):
        k = f'{subset}_{fold}_{exp}_{cfg.lr}_{cfg.reg}'
        for metric in ['accuracy', 'precision', 'recall', 'f1_score', 'auc']:
            vals = metrics.get(f'{metric}_{k}', [])
            metrics_last[metric].extend(vals)

In [ ]:
print('=' * 55)
print('OVERALL PERFORMANCE SUMMARY')
print('=' * 55)
for metric in ['accuracy', 'precision', 'recall', 'f1_score', 'auc']:
    vals = np.array(metrics_last.get(metric, []), dtype=float)
    vals = vals[np.isfinite(vals)]
    label = 'AUC' if metric == 'auc' else metric.capitalize()
    if vals.size:
        print(f'  {label:12s}: {vals.mean():.4f} \u00b1 {vals.std():.4f}  (n={vals.size})')
    else:
        print(f'  {label:12s}: no data')
print('=' * 55)

In [ ]:
# Training history for fold 0, experiment 0
_fold, _exp = cfg.folds[0], 0
_subset = max(cfg.dataset_sizes[_fold])
_k = f'{_subset}_{_fold}_{_exp}_{cfg.lr}_{cfg.reg}'
_hist_list = metrics.get(f'history_{_k}', [])

if _hist_list and isinstance(_hist_list[0], dict):
    plot_training_history(_hist_list[0], cfg.base_key(_fold, _subset), _exp)
    _fname = os.path.join(OUTPUT_DIR, f'training_history_f{_fold}_e{_exp}.pdf')
    plt.savefig(_fname, bbox_inches='tight')
    plt.close()
    print('Saved:', _fname)
else:
    print('No history data found for fold 0 / experiment 0.')

In [ ]:
# Metric distributions across all folds & experiments
METRICS_TO_PLOT = ['accuracy', 'precision', 'recall', 'f1_score', 'auc']
fig, axes = plt.subplots(1, len(METRICS_TO_PLOT),
                         figsize=(4.0 * len(METRICS_TO_PLOT), 3.0))

for ax, metric in zip(axes, METRICS_TO_PLOT):
    vals = np.array(metrics_last.get(metric, []), dtype=float)
    vals = vals[np.isfinite(vals)]
    label = 'AUC' if metric == 'auc' else metric.capitalize()

    if vals.size == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(label)
        continue

    p16, p50, p84 = np.percentile(vals, [16, 50, 84])
    mean = vals.mean()

    vmin, vmax = float(vals.min()), float(vals.max())
    if vmin == vmax:
        eps = max(1e-6, abs(vmin) * 1e-3)
        vmin, vmax = vmin - eps, vmax + eps
    edges = np.linspace(vmin, vmax, 21)

    ax.hist(vals, bins=edges, histtype='stepfilled',
            color='gray', edgecolor='black', linewidth=0.7, alpha=0.85)
    ax.axvspan(p16, p84, color='lightgreen', alpha=0.35, zorder=0, label='68% interval')
    ax.axvline(mean, color='steelblue', linewidth=1.2, label=f'Mean: {mean:.3f}')
    ax.axvline(p50, color='goldenrod', linewidth=1.2, linestyle='--', label=f'Median: {p50:.3f}')
    ax.axvline(p16, color='grey', linewidth=0.7, linestyle=':')
    ax.axvline(p84, color='grey', linewidth=0.7, linestyle=':')
    ax.set_xlabel(label, fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.legend(fontsize=7, framealpha=0.8)

fig.suptitle(f"{cfg.classifier} | {cfg.version} | {cfg.crop_mode}", fontsize=9)
plt.tight_layout()
_fname = os.path.join(OUTPUT_DIR, f'metric_distributions_{cfg.classifier}_{cfg.version}_{cfg.crop_mode}.pdf')
plt.savefig(_fname, bbox_inches='tight')
plt.close()
print('Saved:', _fname)

In [ ]:
# Average confusion matrix (saves to /tmp and displays inline)
_save_dir = '/tmp/dcreclass_nb'
os.makedirs(_save_dir, exist_ok=True)

_largest_sz = max(sz for sizes in cfg.dataset_sizes.values() for sz in sizes)

plot_avg_std_confusion_matrix(
    metrics, dict(metrics_last),
    cfg.galaxy_classes, cfg.classifier,
    cfg.version, _largest_sz,
    [cfg.lr], [cfg.reg],
    cfg.folds, cfg.dataset_sizes,
    cfg.crop_size, cfg.downsample_size,
    cfg.percentile_lo, cfg.percentile_hi,
    num_experiments=cfg.num_experiments,
    save_dir=_save_dir,
)

# Display the saved figure inline
from IPython.display import Image, display
import glob
for _fp in sorted(glob.glob(os.path.join(_save_dir, '*.pdf'))):
    print('Saved:', _fp)

## 3. Comparative ROC curves

Use `plot_comparative_roc` to overlay average ROC curves from different runs on one plot.
Each run is specified as a `(label, RunConfig)` pair. Vary any field between runs.

In [ ]:
def plot_comparative_roc(
    run_configs: List[Tuple[str, RunConfig]],
    subset_size: Optional[int] = None,
    ax: Optional[plt.Axes] = None,
    figsize: Tuple[float, float] = (5.5, 5.5),
    colours: Optional[List[str]] = None,
    title: Optional[str] = None,
) -> plt.Figure:
    """Plot average ROC curves for multiple runs on the same axes.

    Parameters
    ----------
    run_configs : list of (label, RunConfig) pairs.
    subset_size : dataset subset size to use for all runs.
                  Defaults to the largest size in each config's dataset_sizes.
    ax          : existing Axes to draw on; creates a new figure if None.
    figsize     : size of new figure (ignored if ax is provided).
    colours     : colour per run (cycles through defaults if None).
    title       : optional figure title.

    Returns
    -------
    fig : the matplotlib Figure.
    """
    DEFAULT_COLOURS = ['steelblue', 'goldenrod', 'tomato',
                       'mediumpurple', 'darkcyan', 'coral', 'olivedrab']
    if colours is None:
        colours = DEFAULT_COLOURS

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.get_figure()

    fpr_grid = np.linspace(0, 1, 1000)

    for cfg_idx, (label, cfg) in enumerate(run_configs):
        colour = colours[cfg_idx % len(colours)]

        # Determine subset size to use
        _all_sizes = sorted({sz for fs in cfg.dataset_sizes.values() for sz in fs})
        _ss = subset_size if subset_size is not None else _all_sizes[-1]

        min_class     = min(cfg.galaxy_classes)
        n_classes     = len(cfg.galaxy_classes)
        adj_classes   = list(range(n_classes))
        roc_values    = {c: [] for c in adj_classes}  # accumulate TPR curves

        n_loaded = 0
        for experiment, fold in itertools.product(range(cfg.num_experiments), cfg.folds):
            if _ss not in cfg.dataset_sizes.get(fold, []):
                continue

            path = cfg.pkl_path(fold, _ss, experiment)
            try:
                with open(path, 'rb') as fh:
                    data = pickle.load(fh)
            except FileNotFoundError:
                continue

            base = cfg.base_key(fold, _ss)
            y_true  = data['all_true_labels'].get(base)
            y_probs = data['all_pred_probs'].get(base)

            if y_true is None or y_probs is None or len(y_true) == 0:
                continue

            y_true  = np.asarray(y_true)
            y_probs = np.asarray(y_probs)

            # Remap class tags to 0-based indices
            if y_true.max() > n_classes - 1:
                tag_to_idx = {tag: i for i, tag in enumerate(sorted(cfg.galaxy_classes))}
                y_true = np.vectorize(tag_to_idx.get)(y_true)

            if np.unique(y_true).size < 2:
                continue

            if n_classes == 2:
                scores = (y_probs[:, 1] if y_probs.ndim == 2 and y_probs.shape[1] > 1
                          else y_probs.ravel())
                fpr_, tpr_, _ = roc_curve(y_true, scores, pos_label=1)
                roc_values[1].append(np.interp(fpr_grid, fpr_, tpr_))
            else:
                y_bin = label_binarize(y_true, classes=adj_classes)
                for i, cls in enumerate(adj_classes):
                    fpr_, tpr_, _ = roc_curve(y_bin[:, i], y_probs[:, i])
                    roc_values[cls].append(np.interp(fpr_grid, fpr_, tpr_))

            n_loaded += 1

        if n_loaded == 0:
            print(f'  [{label}] No data found — check metrics_dir and config.')
            continue

        # One curve per class (for binary: just class 1)
        plot_classes = [1] if n_classes == 2 else adj_classes
        for cls in plot_classes:
            if not roc_values[cls]:
                continue
            tpr_arr  = np.array(roc_values[cls])
            mean_tpr = np.mean(tpr_arr, axis=0)
            p16_tpr, p84_tpr = np.percentile(tpr_arr, [16, 84], axis=0)

            mean_auc  = auc(fpr_grid, mean_tpr)
            auc_arr   = [auc(fpr_grid, tpr_arr[i]) for i in range(len(tpr_arr))]
            p16_auc, p84_auc = np.percentile(auc_arr, [16, 84])

            legend = (f'{label}  '
                      f'(AUC={mean_auc:.3f}, [{p16_auc:.3f}\u2013{p84_auc:.3f}], n={len(tpr_arr)})')

            ax.plot(fpr_grid, mean_tpr, color=colour, linewidth=1.4, label=legend)
            ax.fill_between(fpr_grid,
                            np.clip(p16_tpr, 0, 1),
                            np.clip(p84_tpr, 0, 1),
                            color=colour, alpha=0.15)
            ax.plot(fpr_grid, np.clip(p16_tpr, 0, 1), color=colour, linewidth=0.5, linestyle=':')
            ax.plot(fpr_grid, np.clip(p84_tpr, 0, 1), color=colour, linewidth=0.5, linestyle=':')

    ax.plot([0, 1], [0, 1], color='black', linewidth=0.8, linestyle='--',
            label='Random classifier')
    ax.set_xlim(0, 1);  ax.set_ylim(0, 1.05)
    ax.set_xlabel('False Positive Rate', fontsize=9)
    ax.set_ylabel('True Positive Rate',  fontsize=9)
    ax.legend(loc='lower right', fontsize=8, framealpha=0.9)
    if title:
        ax.set_title(title, fontsize=9)
    fig.tight_layout()
    return fig

print('plot_comparative_roc() defined')

### Example: compare crop modes on RAW data

Edit the list below to vary any `RunConfig` field between runs.

In [ ]:
# ── Define runs to compare ────────────────────────────────────────────────────
# Each entry: (label, RunConfig)  — only vary the fields you want to compare;
# everything else defaults to the script-04 defaults.

runs_to_compare = [
    ('beam_crop', RunConfig(
        crop_mode       = 'beam_crop',
        version         = 'RAW',
        folds           = list(range(10)),
        num_experiments = 1,
    )),
    ('fov_crop', RunConfig(
        crop_mode       = 'fov_crop',
        version         = 'RAW',
        folds           = list(range(10)),
        num_experiments = 1,
    )),
]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = plot_comparative_roc(
    runs_to_compare,
    title='ROC comparison — RAW data, crop mode varies',
)
_fname = os.path.join(OUTPUT_DIR, 'roc_comparison.pdf')
fig.savefig(_fname, bbox_inches='tight')
plt.close()
print('Saved:', _fname)

In [ ]:
# Optional: save the figure
# fig.savefig('/users/mbredber/scratch/figures/classifying/roc_comparison.pdf',
#             bbox_inches='tight')

## 4. Grouped accuracy bar chart

`plot_accuracy_bars` produces a grouped bar chart where:
- **x-axis** — one group per value of `x_vary` (e.g. different classifiers or data versions)
- **bar colours** — one bar per value of `color_vary` (e.g. crop mode or blur method)
- **bar height / error bar** — mean ± std of `metric` aggregated across all folds and experiments

Both axes are fully configurable; only vary the two dimensions you care about and hold everything else fixed in `base_cfg`.

In [ ]:
import dataclasses
from matplotlib.patches import Patch

def plot_accuracy_bars(
    base_cfg: RunConfig,
    x_vary: str,
    x_values: list,
    metric: str = 'accuracy',
    # optional 3rd dimension — physically adjacent sub-bars (distinguished by hatch)
    adjacent_vary: Optional[str] = None,
    adjacent_values: Optional[list] = None,
    adjacent_labels: Optional[List[str]] = None,
    # optional 4th dimension — overlaid semi-transparent bars (distinguished by colour)
    overlay_vary: Optional[str] = None,
    overlay_values: Optional[list] = None,
    overlay_labels: Optional[List[str]] = None,
    x_labels: Optional[List[str]] = None,
    ax: Optional[plt.Axes] = None,
    figsize: Tuple[float, float] = (9, 4.5),
    colours: Optional[List[str]] = None,
    title: Optional[str] = None,
) -> plt.Figure:
    """Grouped bar chart comparing a metric across up to four config dimensions.

    Dimensions
    ----------
    1. x_vary        (required) — main x-axis groups, e.g. 'classifier' or 'version'.
    2. metric / y    (required) — scalar metric: 'accuracy', 'precision', 'recall',
                                  'f1_score', or 'auc'.
    3. adjacent_vary (optional) — sub-bars placed side-by-side within each x group,
                                  distinguished by hatch pattern, e.g. 'blur_method'.
    4. overlay_vary  (optional) — bars drawn at the same position, semi-transparent
                                  (alpha=0.7) and distinguished by colour, e.g. 'crop_mode'.

    When only one optional dimension is used:
      - adjacent only  →  grouped side-by-side bars, all same colour with different hatches
      - overlay only   →  overlapping bars at each x tick, different colours, alpha 0.7

    Parameters
    ----------
    base_cfg        : RunConfig with all fixed parameters.
    x_vary          : RunConfig field name to place on the x-axis.
    x_values        : Values for that field (one group per value).
    metric          : Metric to plot on the y-axis.
    adjacent_vary   : RunConfig field name for the adjacent (side-by-side) dimension.
    adjacent_values : Values for the adjacent field.
    adjacent_labels : Display labels for adjacent_values (defaults to str(v)).
    overlay_vary    : RunConfig field name for the overlaid dimension.
    overlay_values  : Values for the overlay field.
    overlay_labels  : Display labels for overlay_values (defaults to str(v)).
    x_labels        : Display labels for x_values (defaults to str(v)).
    ax              : Existing Axes to draw on; creates a new figure if None.
    figsize         : Size of new figure (ignored when ax is provided).
    colours         : Colour cycle for the overlay dimension (or adjacent if no overlay).
    title           : Optional figure title.

    Returns
    -------
    fig : the matplotlib Figure.
    """
    COLOURS = colours or [
        'steelblue', 'goldenrod', 'tomato',
        'mediumpurple', 'darkcyan', 'coral', 'olivedrab', '#e377c2',
    ]
    # Hatch patterns distinguish adjacent bars
    HATCHES = ['', '///', '...', 'xxx', '+++', 'ooo', '\\\\\\']

    # Normalise optional dimension specs
    _adj_vals = adjacent_values if adjacent_vary else [None]
    _adj_labs = (adjacent_labels if adjacent_labels is not None
                 else ([str(v) for v in _adj_vals] if adjacent_vary else []))
    _ov_vals  = overlay_values  if overlay_vary  else [None]
    _ov_labs  = (overlay_labels if overlay_labels is not None
                 else ([str(v) for v in _ov_vals] if overlay_vary else []))
    if x_labels is None:
        x_labels = [str(v) for v in x_values]

    n_x   = len(x_values)
    n_adj = len(_adj_vals)
    n_ov  = len(_ov_vals)

    # Bar geometry
    group_width = 0.78
    sub_width   = group_width / n_adj   # width allocated per adjacent slot
    bar_width   = sub_width * 0.84      # actual drawn bar (gap between adjacent groups)
    x_pos       = np.arange(n_x, dtype=float)

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.get_figure()

    # ── Helper: load mean ± std for one (x_val, adj_val, ov_val) combination ──
    def _load_mean_std(x_val, adj_val, ov_val):
        kw = {x_vary: x_val}
        if adjacent_vary and adj_val is not None:
            kw[adjacent_vary] = adj_val
        if overlay_vary and ov_val is not None:
            kw[overlay_vary] = ov_val
        cfg = dataclasses.replace(base_cfg, **kw)
        run = load_run(cfg, verbose=False)
        vals = []
        for fold in cfg.folds:
            subset = max(cfg.dataset_sizes[fold])
            for exp in range(cfg.num_experiments):
                k = f'{metric}_{subset}_{fold}_{exp}_{cfg.lr}_{cfg.reg}'
                vals.extend(v for v in run.get(k, []) if np.isfinite(v))
        arr = np.array(vals, dtype=float)
        return (arr.mean() if arr.size else np.nan,
                arr.std()  if arr.size else np.nan)

    # ── Draw bars ─────────────────────────────────────────────────────────────
    # Colour cycles: overlay uses COLOURS; if no overlay, adjacent uses COLOURS
    alpha_bar = 0.70 if n_ov > 1 else 0.85

    for adj_idx, adj_val in enumerate(_adj_vals):
        adj_offset = sub_width * (adj_idx - (n_adj - 1) / 2)
        bar_x      = x_pos + adj_offset
        hatch      = HATCHES[adj_idx % len(HATCHES)]

        # When only adjacent is active (no overlay) use COLOURS for colour
        sole_colour = COLOURS[adj_idx % len(COLOURS)] if not overlay_vary else None

        for ov_idx, ov_val in enumerate(_ov_vals):
            colour = sole_colour if sole_colour else COLOURS[ov_idx % len(COLOURS)]

            means, stds = zip(*[
                _load_mean_std(xv, adj_val, ov_val) for xv in x_values
            ])
            means = np.array(means, dtype=float)
            stds  = np.array(stds,  dtype=float)

            # Replace nan means with 0 for display (no bar drawn)
            display_means = np.where(np.isnan(means), 0.0, means)
            display_stds  = np.where(np.isnan(stds),  np.nan, stds)

            # Legend label: add once per overlay value (not per adj group)
            ov_legend_lbl = _ov_labs[ov_idx] if (overlay_vary and adj_idx == 0) else '_nolegend_'
            adj_legend_lbl = _adj_labs[adj_idx] if (adjacent_vary and ov_idx == 0) else '_nolegend_'

            ax.bar(
                bar_x, display_means, bar_width,
                yerr=display_stds, capsize=3,
                color=colour, alpha=alpha_bar,
                hatch=hatch,
                label=ov_legend_lbl,
                error_kw=dict(linewidth=0.9, ecolor='#111111', capthick=0.9),
                zorder=2,
            )

    # ── Sub-tick labels for adjacent dimension ────────────────────────────────
    if adjacent_vary and n_adj > 1:
        fig.canvas.draw()           # needed for accurate ylim before text placement
        y_lo, y_hi = ax.get_ylim()
        y_sub = y_lo - 0.055 * (y_hi - y_lo)
        for adj_idx, adj_lab in enumerate(_adj_labs):
            adj_offset = sub_width * (adj_idx - (n_adj - 1) / 2)
            for xi in range(n_x):
                ax.text(xi + adj_offset, y_sub, adj_lab,
                        ha='center', va='top', fontsize=6.5,
                        color='#333333', clip_on=False)
        fig.subplots_adjust(bottom=0.18)

    # ── Axes formatting ───────────────────────────────────────────────────────
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, fontsize=9)
    y_label = 'AUC' if metric == 'auc' else metric.capitalize()
    ax.set_ylabel(y_label, fontsize=9)
    ax.set_ylim(0, 1.08)
    ax.axhline(0.5, color='grey', linewidth=0.7, linestyle='--',
               label='Random (0.5)', zorder=1)

    # ── Legend(s) ─────────────────────────────────────────────────────────────
    ov_title = overlay_vary or (adjacent_vary or 'condition')
    main_legend = ax.legend(title=ov_title, fontsize=8, title_fontsize=8,
                            framealpha=0.9, loc='lower right')

    if adjacent_vary and n_adj > 1:
        adj_handles = [
            Patch(facecolor='#cccccc', edgecolor='#333333',
                  hatch=HATCHES[i % len(HATCHES)], label=lab, linewidth=0.8)
            for i, lab in enumerate(_adj_labs)
        ]
        ax.add_artist(main_legend)
        ax.legend(handles=adj_handles, title=adjacent_vary, fontsize=8,
                  title_fontsize=8, framealpha=0.9, loc='upper left')

    if title:
        ax.set_title(title, fontsize=9)

    fig.tight_layout()
    return fig

print('plot_accuracy_bars() defined  (2-axis to 4-axis)')

In [ ]:
# ── Edit the call below to your comparison of interest ───────────────────────
#
# Dimension guide:
#   x_vary        → main x-axis grouping (required)
#   metric        → y-axis metric (required)
#   adjacent_vary → sub-bars side-by-side, labelled below ticks, hatch-patterned (optional)
#   overlay_vary  → bars overlaid at the same position, semi-transparent (optional)
#
# Any RunConfig field can go on any axis:
#   'classifier', 'version', 'crop_mode', 'blur_method', 'lr', 'reg', ...
#
# Example A — 2 axes: versions on x, accuracy on y
# fig = plot_accuracy_bars(
#     base_cfg     = RunConfig(folds=list(range(10))),
#     x_vary       = 'version',
#     x_values     = ['RAW', 'T25kpc', 'T50kpc', 'T100kpc'],
#     metric       = 'accuracy',
# )
#
# Example B — 3 axes: versions on x, crop_mode overlaid
# fig = plot_accuracy_bars(
#     base_cfg      = RunConfig(folds=list(range(10))),
#     x_vary        = 'version',
#     x_values      = ['RAW', 'T25kpc', 'T50kpc', 'T100kpc'],
#     overlay_vary  = 'crop_mode',
#     overlay_values = ['beam_crop', 'fov_crop'],
#     metric        = 'accuracy',
# )
#
# Example C — 4 axes: versions on x, blur_method adjacent, crop_mode overlaid
fig = plot_accuracy_bars(
    base_cfg         = RunConfig(
        folds           = list(range(10)),
        num_experiments = 1,
    ),
    x_vary           = 'version',
    x_values         = ['RAW', 'T25kpc', 'T50kpc', 'T100kpc'],
    adjacent_vary    = 'blur_method',
    adjacent_values  = ['circular', 'cheat'],
    overlay_vary     = 'crop_mode',
    overlay_values   = ['beam_crop', 'fov_crop'],
    metric           = 'accuracy',
    title            = 'Accuracy: tapering scale (x), blur method (hatch), crop mode (colour)',
)

_fname = os.path.join(OUTPUT_DIR, 'accuracy_bars_version_blur_crop.pdf')
fig.savefig(_fname, bbox_inches='tight')
plt.close()
print('Saved:', _fname)